# 🚀 OMEGA Cloud LLM GPU Runner (Google Colab Edition)
### Uncensored High-Performance Inference for Terminal AI Agents
Supports **DeepSeek V4 Pro 9B Uncensored** & **Qwen 3.8 27B Turbo Heretic** with full CUDA GPU acceleration.

In [ ]:
# @title ⚙️ 1. Check GPU & Install Runtime Tools
# @markdown Run this cell first to verify GPU access and install Ollama + Cloudflare Tunnel.
import subprocess, os, sys, time, re

print("🔍 Checking GPU hardware...")
gpu = subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
if "failed" in gpu.lower() or not gpu.strip():
    raise RuntimeError("❌ No GPU found! Please go to Runtime -> Change runtime type -> Select T4 GPU and restart.")
print(f"✅ Active GPU: {gpu.strip()}")

print("📦 Installing Ollama with CUDA acceleration...")
os.system("curl -fsSL https://ollama.com/install.sh | sh >/dev/null 2>&1")

print("☁️ Installing Cloudflare Tunnel...")
os.system("curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 >/dev/null 2>&1")
os.system("chmod +x /usr/local/bin/cloudflared")
os.system("pip install -q huggingface_hub")

print("✅ System setup complete!")


In [ ]:
# @title 🧠 2. Select Model & Start GPU Server
# @markdown Choose which uncensored model you want to deploy on GPU:
MODEL_CHOICE = "DeepSeek-V4-Pro-9B-UNCENSORED (Fast, Recommended)" # @param ["DeepSeek-V4-Pro-9B-UNCENSORED (Fast, Recommended)", "Qwen3.8-27B-TURBO-Heretic-NEO-CODER"]

import os, subprocess, time, re
from huggingface_hub import hf_hub_download

os.makedirs("/content/models", exist_ok=True)

if "9B" in MODEL_CHOICE:
    REPO_ID = "xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic"
    FILENAME = "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-Q4_K_M.gguf"
    MODEL_NAME = "deepseek-heretic-fast"
    TEMPLATE = "chatml"
else:
    REPO_ID = "DavidAU/Qwen3.8-27B-TURBO-Fable-Cold-Fusion-735-882-Heretic-Uncensored-NEO-CODER-MAX-MTP-GGUF"
    FILENAME = "Qwen3.8-27B-TURBO-Fable-Cold-Fusion-735-882-Heretic-Uncensored-NEO-CODER-MAX-MTP-Q3_K_M.gguf"
    MODEL_NAME = "qwen-27b-turbo"
    TEMPLATE = "chatml"

print(f"⬇️ Downloading {FILENAME} from Hugging Face...")
gguf_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME, local_dir="/content/models")
print(f"✅ Model weights ready: {gguf_path} ({os.path.getsize(gguf_path) / (1024**3):.2f} GB)")

# Kill any existing server
os.system("pkill -f ollama 2>/dev/null; pkill -f cloudflared 2>/dev/null")
time.sleep(2)

# Start Ollama Daemon
print("⚡ Starting Ollama CUDA daemon...")
os.system("nohup ollama serve > /content/ollama.log 2>&1 &")
time.sleep(4)

# Create Ollama model from GGUF
print(f"🔧 Creating model {MODEL_NAME}...")
modelfile_content = f"FROM {gguf_path}
PARAMETER num_ctx 16384
PARAMETER temperature 0.7
"
with open("/content/Modelfile", "w") as f:
    f.write(modelfile_content)

os.system(f"ollama create {MODEL_NAME} -f /content/Modelfile")
print(f"✅ Model {MODEL_NAME} created and loaded into Ollama!")

# Start Cloudflare Tunnel
print("🌐 Starting Cloudflare Quick Tunnel...")
os.system("nohup cloudflared tunnel --url http://127.0.0.1:11434 > /content/tunnel.log 2>&1 &")

tunnel_url = None
for _ in range(40):
    time.sleep(2)
    if os.path.exists("/content/tunnel.log"):
        with open("/content/tunnel.log", "r") as f:
            c = f.read()
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", c)
            if m:
                tunnel_url = m.group(0)
                break

if tunnel_url:
    print("
" + "="*65)
    print("🔥 GOOGLE COLAB GPU ENDPOINT IS LIVE! 🔥")
    print("="*65)
    print(f"🔗 Public Tunnel URL: {tunnel_url}")
    print(f"🤖 Model Name:        {MODEL_NAME}")
    print("="*65)
    print("
📱 RUN THIS COMMAND IN TERMUX TO CONNECT ALL AGENTS:")
    print(f"link-colab {tunnel_url}")
    print("="*65)
else:
    print("❌ Failed to start Cloudflare tunnel. Check /content/tunnel.log")
